# RLC Circuit Analysis with KoopSim

## Series RLC Circuit

The **series RLC circuit** is a fundamental linear system governed by:

$$\dot{q} = i$$
$$\dot{i} = -\frac{R}{L} i - \frac{1}{LC} q$$

where $q$ is the capacitor charge and $i = \dot{q}$ is the current. The state vector is $[q, i]$.

## Why This is a Perfect Koopman Test Case

Since the RLC circuit is **linear**, the Koopman operator in the original state space *is* the state transition matrix -- no dictionary lifting is needed. The identity dictionary should achieve **perfect recovery**.

The eigenvalues of the Koopman matrix directly encode:
- The **damped oscillation frequency** $\omega_d = \sqrt{\frac{1}{LC} - \left(\frac{R}{2L}\right)^2}$
- The **decay rate** $\alpha = \frac{R}{2L}$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from koopsim import KoopSim
from koopsim.systems import RLCCircuit
from koopsim.utils.visualization import (
    plot_trajectory_comparison,
    plot_eigenspectrum,
)

## 1. System Setup and Analytical Solution

We use $R=1.0$ $\Omega$, $L=1.0$ H, $C=1.0$ F. This gives an underdamped system with:
- Decay rate: $\alpha = R/(2L) = 0.5$
- Natural frequency: $\omega_0 = 1/\sqrt{LC} = 1.0$ rad/s
- Damped frequency: $\omega_d = \sqrt{\omega_0^2 - \alpha^2} = \sqrt{0.75} \approx 0.866$ rad/s

In [ ]:
R, L, C = 1.0, 1.0, 1.0
circuit = RLCCircuit(R=R, L=L, C=C)
print(f"System: {circuit.name}, state dimension: {circuit.state_dim}")

# Analytical parameters
alpha = R / (2 * L)
omega_0 = 1.0 / np.sqrt(L * C)
omega_d = np.sqrt(omega_0**2 - alpha**2)
print(f"Decay rate (alpha): {alpha:.4f}")
print(f"Natural frequency (omega_0): {omega_0:.4f} rad/s")
print(f"Damped frequency (omega_d):  {omega_d:.4f} rad/s")

## 2. Generate Training Data

In [ ]:
x0 = np.array([1.0, 0.0])  # initial charge q=1, current i=0
dt = 0.01

X_train, Y_train = circuit.generate_snapshots(
    x0, dt=dt, n_steps=300, n_trajectories=10
)
print(f"Training data: {X_train.shape[0]} snapshot pairs")

## 3. Train EDMD with Identity Dictionary

For a linear system, the identity dictionary suffices. No polynomial or RBF lifting needed.

In [ ]:
# No poly_degree or rbf_centers -> defaults to identity dictionary
sim = KoopSim(method="edmd")
sim.fit(X_train, Y_train, dt=dt)
print(sim)
print(f"Koopman matrix shape: {sim.model.get_koopman_matrix().shape}")

## 4. Predict and Compare to Analytical Solution

The analytical solution for $q(0)=1, i(0)=0$ is:

$$q(t) = e^{-\alpha t} \left(\cos(\omega_d t) + \frac{\alpha}{\omega_d} \sin(\omega_d t)\right)$$
$$i(t) = -\frac{\omega_0^2}{\omega_d} e^{-\alpha t} \sin(\omega_d t)$$

In [ ]:
# Analytical solution
x0_test = np.array([1.0, 0.0])
times = np.linspace(0, 15, 500)

q_analytical = np.exp(-alpha * times) * (
    np.cos(omega_d * times) + (alpha / omega_d) * np.sin(omega_d * times)
)
i_analytical = -(omega_0**2 / omega_d) * np.exp(-alpha * times) * np.sin(omega_d * times)
traj_analytical = np.column_stack([q_analytical, i_analytical])

# Koopman prediction
traj_koopman = sim.predict_trajectory(x0_test, times)

# ODE ground truth
traj_ode = circuit.generate_trajectory(x0_test, dt=times[1] - times[0], n_steps=len(times) - 1)

In [ ]:
fig = plot_trajectory_comparison(
    times, traj_analytical, traj_koopman,
    labels=["Charge (q)", "Current (i)"]
)
fig.suptitle("RLC Circuit: Analytical vs Koopman", y=1.01)
plt.show()

## 5. Perfect Recovery for Linear Systems

For a linear system, the Koopman matrix K is exactly the state transition matrix $e^{A \cdot dt}$, where $A$ is the system matrix. Let's verify the prediction error is near machine precision.

In [ ]:
# One-step validation
X_val, Y_val = circuit.generate_snapshots(
    np.array([0.5, -0.3]), dt=dt, n_steps=100, n_trajectories=5
)
rmse = sim.validate(X_val, Y_val, metric="rmse")
rel_err = sim.validate(X_val, Y_val, metric="relative")
print(f"One-step RMSE:      {rmse:.6e}")
print(f"Relative error:     {rel_err:.6e}")

# Multi-step trajectory error
error_koopman = np.sqrt(np.mean((traj_analytical - traj_koopman)**2, axis=1))
print(f"\nTrajectory error (Koopman vs analytical):")
print(f"  Mean RMSE:  {np.mean(error_koopman):.6e}")
print(f"  Max RMSE:   {np.max(error_koopman):.6e}")

## 6. Eigenspectrum: Damped Oscillation Frequency

The Koopman eigenvalues $\lambda$ are related to the continuous-time eigenvalues $s$ by $\lambda = e^{s \cdot dt}$. For the RLC circuit, $s = -\alpha \pm j\omega_d$.

In [ ]:
fig = plot_eigenspectrum(sim.model)
plt.show()

In [ ]:
spectral = sim.spectral_analysis()
eigenvalues = spectral["eigenvalues"]

print("Koopman eigenvalues:")
for i, lam in enumerate(eigenvalues):
    print(f"  lambda_{i} = {lam:.6f}  (|lambda| = {abs(lam):.6f})")

# Extract continuous-time eigenvalues: s = log(lambda) / dt
s_values = np.log(eigenvalues) / dt
print(f"\nContinuous-time eigenvalues (s = log(lambda)/dt):")
for i, s in enumerate(s_values):
    print(f"  s_{i} = {s:.4f}")

# Compare to analytical
print(f"\nAnalytical eigenvalues: s = {-alpha:.4f} +/- {omega_d:.4f}j")
print(f"Identified:")
for s in s_values:
    print(f"  Re(s) = {s.real:.4f} (expected {-alpha:.4f}), "
          f"Im(s) = {s.imag:.4f} (expected +/-{omega_d:.4f})")

## 7. Compare Koopman Matrix to Analytical Transition Matrix

For verification, we compare the learned K to the analytical $e^{A \cdot dt}$.

In [ ]:
from scipy.linalg import expm

# System matrix
A_system = np.array([
    [0.0, 1.0],
    [-1.0 / (L * C), -R / L]
])

# Analytical transition matrix
K_analytical = expm(A_system * dt)
K_learned = sim.model.get_koopman_matrix()

print("Analytical K (expm(A*dt)):")
print(K_analytical)
print(f"\nLearned K:")
print(K_learned)
print(f"\nDifference (Frobenius norm): {np.linalg.norm(K_learned - K_analytical, 'fro'):.6e}")

## 8. Long-Term Prediction

Because the Koopman approach uses `expm(L * t)` rather than time-stepping, we can predict at arbitrarily far future times with a single matrix operation.

In [ ]:
# Predict far into the future
t_far = np.array([50.0, 100.0, 500.0, 1000.0])
for t in t_far:
    state = sim.predict(x0_test, t=t)
    q_true = np.exp(-alpha * t) * (
        np.cos(omega_d * t) + (alpha / omega_d) * np.sin(omega_d * t)
    )
    print(f"t={t:7.0f}: q_pred={state[0]:+.6e}, q_analytical={q_true:+.6e}, "
          f"error={abs(state[0] - q_true):.2e}")

## Summary

In this notebook we:
1. Modeled a series RLC circuit ($R=1\Omega$, $L=1$H, $C=1$F)
2. Trained EDMD with the identity dictionary (appropriate for a linear system)
3. Achieved **near-perfect recovery**: the Koopman matrix matches the analytical transition matrix
4. Verified predictions against the closed-form analytical solution
5. Extracted the damped oscillation frequency ($\omega_d \approx 0.866$ rad/s) and decay rate ($\alpha = 0.5$) from the Koopman eigenspectrum
6. Demonstrated instant long-range predictions via matrix exponential

The RLC circuit demonstrates that for **linear systems**, Koopman operator learning recovers the exact dynamics. This serves as a useful sanity check and baseline before applying the method to nonlinear systems.